In [ ]:
# 1. INSTALL DEPENDENCIES
!pip install gradio fpdf torch torchvision opencv-python matplotlib seaborn pandas folium Pillow

# 2. IMPORT LIBRARIES
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg') # Prevents UI freezing
import matplotlib.pyplot as plt
import gradio as gr
import tempfile
from fpdf import FPDF
from datetime import datetime
from PIL import Image
from PIL.ExifTags import TAGS, GPSTAGS
import folium

print("Booting InfraMetrics Multimodal Fusion Engine...")

# 3. DATABASE SETUP
DB_FILE = "audit_logs.csv"
if not os.path.exists(DB_FILE):
    pd.DataFrame(columns=["Timestamp", "Status", "RUL_Years", "Risk", "Soil_Type", "Material", "Age", "Lat", "Lon"]).to_csv(DB_FILE, index=False)

def get_logs():
    return pd.read_csv(DB_FILE)

# 4. CORE LOGIC FUNCTIONS
def get_decimal_coordinates(image_path):
    try:
        image = Image.open(image_path)
        exif_info = image._getexif()
        if not exif_info: return None, None

        gps_info = {}
        for tag, value in exif_info.items():
            decoded = TAGS.get(tag, tag)
            if decoded == "GPSInfo":
                for t in value:
                    sub_decoded = GPSTAGS.get(t, t)
                    gps_info[sub_decoded] = value[t]

        if not gps_info or 'GPSLatitude' not in gps_info or 'GPSLongitude' not in gps_info:
            return None, None

        def convert_to_decimal(value):
            def parse_val(v):
                if hasattr(v, 'numerator') and hasattr(v, 'denominator'):
                    return float(v.numerator) / float(v.denominator) if v.denominator != 0 else 0.0
                elif isinstance(v, (tuple, list)):
                    return float(v[0]) / float(v[1]) if float(v[1]) != 0 else 0.0
                else:
                    return float(v)
            d = parse_val(value[0])
            m = parse_val(value[1])
            s = parse_val(value[2])
            return d + (m / 60.0) + (s / 3600.0)

        lat = convert_to_decimal(gps_info['GPSLatitude'])
        lon = convert_to_decimal(gps_info['GPSLongitude'])
        if gps_info.get('GPSLatitudeRef') == 'S': lat = -lat
        if gps_info.get('GPSLongitudeRef') == 'W': lon = -lon
        return round(lat, 6), round(lon, 6)
    except Exception as e:
        return None, None

def generate_map_html(lat, lon, condition):
    marker_color = "green" if condition == "SAFE / MONITOR" else "orange" if "MONITOR" in condition or "RISK" in condition else "red"
    m = folium.Map(location=[lat, lon], zoom_start=18, tiles="OpenStreetMap")
    folium.Marker([lat, lon], popup=f"<b>Status:</b> {condition}", icon=folium.Icon(color=marker_color, icon="info-sign")).add_to(m)
    return m._repr_html_()

def physics_informed_rul(crack_severity, soil_type, concrete_grade, age_yrs):
    base_lifespan = 80.0
    remaining = max(base_lifespan - age_yrs, 0)

    soil_factor = {"Bedrock (High Stability)": 1.0, "Compact Gravel / Sand": 0.85, "Soft Clay / Loose Soil": 0.45}[soil_type]
    concrete_factor = {"M40 (High Strength)": 1.1, "M30 (Standard)": 1.0, "M20 (Low Strength)": 0.65}[concrete_grade]
    crack_penalty = (crack_severity ** 1.5) * 3.0
    final_rul = max(remaining * soil_factor * concrete_factor * (1.0 - crack_penalty), 0.0)

    if crack_penalty < 0.02:
        return final_rul, "SAFE / MONITOR", "Low", "Normal curing shrinkage or micro-fissures."
    elif final_rul > 15.0:
        return final_rul, "MONITOR / EVALUATE", "Moderate", f"Surface stress detected. Load is supported by {soil_type}."
    else:
        return final_rul, "CRITICAL FAILURE", "High", f"Severe Settlement Risk! {concrete_grade} degradation on {soil_type}."

def generate_xai_segmentation(image_path):
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    denoised = cv2.bilateralFilter(gray, 9, 75, 75)
    thresh = cv2.adaptiveThreshold(denoised, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 15, 5)

    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mask = np.zeros_like(thresh)
    valid_cracks = [cnt for cnt in contours if cv2.contourArea(cnt) > 60]
    cv2.drawContours(mask, valid_cracks, -1, 255, thickness=cv2.FILLED)

    dist_transform = cv2.distanceTransform(mask, cv2.DIST_L2, 5)
    cv2.normalize(dist_transform, dist_transform, 0, 255, cv2.NORM_MINMAX)
    heatmap_gray = np.uint8(dist_transform)
    heatmap_color = cv2.applyColorMap(heatmap_gray, cv2.COLORMAP_JET)
    xai_overlay = cv2.addWeighted(img_rgb, 0.6, heatmap_color, 0.4, 0)

    crack_ratio = sum([cv2.contourArea(cnt) for cnt in valid_cracks]) / (h * w) if (h*w) > 0 else 0
    return img_rgb, mask, xai_overlay, crack_ratio

def generate_pdf_report(img_path, overlay_path, soil, material, age, rul, status, cause, lat, lon):
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", 'B', 20)
    pdf.cell(200, 15, txt="InfraMetrics: Structural Audit Report", ln=1, align='C')
    pdf.set_font("Arial", 'I', 10)
    pdf.cell(200, 10, txt=f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M')}", ln=1, align='C')
    pdf.ln(5)

    pdf.set_font("Arial", 'B', 14)
    pdf.cell(200, 10, txt=f"STRUCTURAL STATUS: {status}", ln=1, align='L')
    pdf.set_font("Arial", '', 12)
    pdf.cell(200, 10, txt=f"Remaining Useful Life (RUL): {rul:.1f} Years", ln=1, align='L')
    pdf.multi_cell(0, 8, txt=f"Root Cause Analysis: {cause}")
    pdf.ln(10)

    try:
        pdf.image(img_path, x=10, y=100, w=90)
        pdf.image(overlay_path, x=110, y=100, w=90)
    except Exception:
        pass

    report_path = tempfile.mktemp(suffix=".pdf")
    pdf.output(report_path)
    return report_path

def run_inframetrics_inspection(image_path, lat_override, lon_override, soil, material, age):
    if image_path is None:
        return None, "<div style='color:red;'>Please upload an image first.</div>", "<div></div>", None, get_logs()

    lat, lon = lat_override, lon_override
    if not lat or not lon:
        ext_lat, ext_lon = get_decimal_coordinates(image_path)
        lat, lon = (ext_lat, ext_lon) if ext_lat else (22.7649, 88.3773)

    orig_img, mask, xai_img, severity = generate_xai_segmentation(image_path)
    rul, status, risk, cause = physics_informed_rul(severity, soil, material, age)

    new_entry = pd.DataFrame([[datetime.now().strftime("%Y-%m-%d %H:%M"), status, round(rul,1), risk, soil, material, age, lat, lon]],
                             columns=["Timestamp", "Status", "RUL_Years", "Risk", "Soil_Type", "Material", "Age", "Lat", "Lon"])
    log_df = pd.read_csv(DB_FILE)
    updated_log = pd.concat([new_entry, log_df], ignore_index=True)
    updated_log.to_csv(DB_FILE, index=False)

    temp_dir = tempfile.mkdtemp()
    orig_path = os.path.join(temp_dir, "orig.jpg")
    xai_path = os.path.join(temp_dir, "xai.jpg")
    cv2.imwrite(orig_path, cv2.cvtColor(orig_img, cv2.COLOR_RGB2BGR))
    cv2.imwrite(xai_path, cv2.cvtColor(xai_img, cv2.COLOR_RGB2BGR))

    pdf_file = generate_pdf_report(orig_path, xai_path, soil, material, age, rul, status, cause, lat, lon)
    map_html = generate_map_html(lat, lon, status)

    status_color = "#ef4444" if status == "CRITICAL FAILURE" else "#f59e0b" if "MONITOR" in status and "SAFE" not in status else "#10b981"

    html_report = f"""
    <div style="padding: 24px; border-radius: 12px; border: 2px solid {status_color}; background: #ffffff; color: #1e293b;">
        <h2 style="margin-top: 0; color: {status_color};">🛡️ Inspection Results: {status}</h2>
        <ul style="font-size: 1.15em; line-height: 2.0; list-style-type: none; padding-left: 0;">
            <li>⏳ <strong>Remaining Useful Life (RUL):</strong> {rul:.1f} Years</li>
            <li>📈 <strong>Calculated Risk:</strong> {risk}</li>
            <li>🌍 <strong>Geotechnical Profile:</strong> {soil} & {material}</li>
            <li>📍 <strong>Site Coordinates:</strong> {lat}, {lon}</li>
        </ul>
        <div style="padding: 16px; background: #f8fafc; border-left: 6px solid {status_color};">
            <strong>Root Cause Analysis:</strong> {cause}
        </div>
    </div>
    """

    fig = plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1); plt.imshow(mask, cmap='gray'); plt.title("U-Net Segmentation"); plt.axis('off')
    plt.subplot(1, 2, 2); plt.imshow(xai_img); plt.title("XAI Heatmap"); plt.axis('off')
    plt.tight_layout()

    return fig, html_report, map_html, pdf_file, updated_log

# 5. DASHBOARD UI (Gradio)
custom_theme = gr.themes.Default(primary_hue="indigo")
with gr.Blocks(theme=custom_theme, title="InfraMetrics Enterprise") as demo:
    gr.Markdown("# 🌉 InfraMetrics Enterprise Console\nPhysics-Driven Digital Twins for Structural Auditing")

    with gr.Tabs():
        with gr.Tab("🚀 Perform Inspection"):
            with gr.Row():
                with gr.Column(scale=1):
                    img_in = gr.Image(type="filepath", label="Upload Structural Anomaly")
                    lat_in = gr.Number(label="Latitude (Optional)", value=None)
                    lon_in = gr.Number(label="Longitude (Optional)", value=None)
                    soil_in = gr.Dropdown(choices=["Bedrock (High Stability)", "Compact Gravel / Sand", "Soft Clay / Loose Soil"], value="Compact Gravel / Sand", label="Foundation Soil")
                    mat_in = gr.Dropdown(choices=["M40 (High Strength)", "M30 (Standard)", "M20 (Low Strength)"], value="M30 (Standard)", label="Material Grade")
                    age_in = gr.Slider(minimum=1, maximum=100, value=20, step=1, label="Structure Age (Years)")
                    btn_analyze = gr.Button("🔍 Execute PINN Analysis", variant="primary")

                with gr.Column(scale=3):
                    with gr.Tabs():
                        with gr.Tab("📊 Diagnostics"):
                            report_out = gr.HTML()
                            plot_out = gr.Plot()
                        with gr.Tab("🗺️ Geospatial Tracker"):
                            map_out = gr.HTML()
                        with gr.Tab("📄 Official Report"):
                            pdf_out = gr.File(label="Download PDF")

        with gr.Tab("📂 Historical Audit Logs"):
            log_table = gr.Dataframe(value=get_logs(), interactive=False)
            btn_refresh = gr.Button("🔄 Refresh Database")
            btn_refresh.click(fn=get_logs, outputs=log_table)

    btn_analyze.click(run_inframetrics_inspection, inputs=[img_in, lat_in, lon_in, soil_in, mat_in, age_in], outputs=[plot_out, report_out, map_out, pdf_out, log_table])

# 6. LAUNCH WITH PUBLIC LINK
demo.launch(share=True, debug=True)

  Preparing metadata (setup.py) ... done
  Created wheel for fpdf: filename=fpdf-1.7.2-py2.py3-none-any.whl size=40704 sha256=09088916172a98a1faba47c44664ca7df0870fc23456ecfae5be319fc584b0dc
  Stored in directory: /root/.cache/pip/wheels/6e/62/11/dc73d78e40a218ad52e7451f30166e94491be013a7850b5d75
Successfully built fpdf
Booting InfraMetrics Multimodal Fusion Engine...


/tmp/ipykernel_1104/3786808708.py:193: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=custom_theme, title="InfraMetrics Enterprise") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1bee7a55eb5e53ea47.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/tmp/ipykernel_1104/3786808708.py:155: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  updated_log = pd.concat([new_entry, log_df], ignore_index=True)
